In [ ]:
import torch.nn as nn
import torch

In [ ]:
class InceptionModule(nn.Module):
    def __init__(self, in_channels, n1, n2_reduce, n2, n3_reduce, n3, n4):
        super().__init__()
        self.submodules = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_channels=in_channels, kernel_size=1, stride=1, out_channels=n1),
                nn.ReLU(),
            ),
            nn.Sequential(
                nn.Conv2d(in_channels=in_channels, kernel_size=1, stride=1, out_channels=n2_reduce),
                nn.ReLU(),
                nn.Conv2d(in_channels=n2_reduce, kernel_size=3, stride=1, out_channels=n2, padding="same"),
                nn.ReLU(),
            ),
            nn.Sequential(
                nn.Conv2d(in_channels=in_channels, kernel_size=1, stride=1, out_channels=n3_reduce),
                nn.ReLU(),
                nn.Conv2d(in_channels=n3_reduce, kernel_size=5, stride=1, out_channels=n3, padding="same"),
                nn.ReLU(),
            ),
            nn.Sequential(
                nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
                nn.Conv2d(in_channels=in_channels, out_channels=n4, kernel_size=1, stride=1),
                nn.ReLU(),
            )
        ])

    def forward(self, inputs):
        outputs = [module(inputs) for module in self.submodules]
        return torch.cat(outputs, dim=1)

In [ ]:
class GoogLeNet(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=64, kernel_size=7, stride=2, padding=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=1, stride=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=192, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(192),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )

        self.inception_modules = nn.Sequential(
            InceptionModule(192,  64,  96, 128, 16,  32,  32),  # 64+128+32+32 = 256
            InceptionModule(256, 128, 128, 192, 32,  96,  64),  # 128+192+96+64 = 480
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),

            InceptionModule(480, 192,  96, 208, 16,  48,  64),  # 192+208+48+64 = 512
            InceptionModule(512, 160, 112, 224, 24,  64,  64),  # 160+224+64+64 = 512
            InceptionModule(512, 128, 128, 256, 24,  64,  64),  # 128+256+64+64 = 512
            InceptionModule(512, 112, 144, 288, 32,  64,  64),  # 112+288+64+64 = 528
            InceptionModule(528, 256, 160, 320, 32, 128,  128), # 256+320+128+128 = 832
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),

            InceptionModule(832, 256, 160, 320, 32, 128, 128),  # 256+320+128+128 = 832
            InceptionModule(832, 384, 192, 384, 48, 128, 128),  # 384+384+128+128 = 1024
        )

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(1024, 1000),
        )

    def forward(self, X):
        X = self.stem(X)
        X = self.inception_modules(X)
        X = self.head(X)
        return X